# Assignment 4

**Student roll number:** 1024170115

## Solution 1

In [ ]:
import pandas as pd

roll_number = "1024170115"
last_digits = [int(d) for d in roll_number[-2:]]
categories = ["billing", "account", "general"]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

personalized_templates = {
    "billing": ("how can I download my fee receipt", "Open Payments, choose the transaction, and select Download receipt.", "receipt invoice transaction download"),
    "account": ("how do I update my registered mobile number", "Open Profile Settings, update the mobile number, and verify it with the OTP.", "mobile number profile update otp"),
    "general": ("where can I find the student help desk", "The student help desk is in the main administration building.", "help desk support office location"),
}
personalized = []
for digit in last_digits:
    category = categories[digit % 3]
    question, answer, keywords = personalized_templates[category]
    personalized.append({"question": question, "answer": answer, "keywords": keywords, "category": category})

faq_df = pd.DataFrame(fixed_entries + personalized)
faq_df

## Solution 2
The score combines overlap with the question and keyword fields. It returns every entry, sorted from the strongest match downward.

In [ ]:
import re

def words(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))

def score_query(query, df=faq_df):
    query_words = words(query)
    rows = []
    for idx, row in df.iterrows():
        question_words = words(row["question"])
        keyword_words = words(row["keywords"])
        denominator = max(len(query_words), 1)
        confidence = (2 * len(query_words & question_words) + len(query_words & keyword_words)) / (3 * denominator)
        rows.append({"index": idx, "confidence": confidence, **row.to_dict()})
    return pd.DataFrame(rows).sort_values(["confidence", "index"], ascending=[False, True]).reset_index(drop=True)

score_query("how can I pay the fee").head()

## Solution 3

In [ ]:
def same_category(category_name, df=faq_df):
    return df.loc[df["category"].str.casefold() == category_name.casefold(), ["question", "answer", "keywords", "category"]]

same_category("account")

## Solution 4


In [ ]:
entry_index = 4  # one of the FAQ rows
new_keyword = input("Add one keyword to this FAQ entry: ").strip().lower()
if new_keyword:
    existing = faq_df.at[entry_index, "keywords"].split()
    if new_keyword not in existing:
        existing.append(new_keyword)
    faq_df.at[entry_index, "keywords"] = " ".join(existing)
output_file = f"{roll_number}_faq_data.csv"
faq_df.to_csv(output_file, index=False)
print(f"Saved {len(faq_df)} FAQ rows to {output_file}")
faq_df

## Solution 5

In [ ]:
faq_df.groupby("category").size().rename("entry_count").to_frame()

## Solution 6

In [ ]:
def best_matches(query, df=faq_df):
    ranked = score_query(query, df)
    if ranked.empty or ranked["confidence"].max() == 0:
        return ranked.iloc[0:0]
    top = ranked["confidence"].max()
    return ranked.loc[ranked["confidence"] == top]

print("Tie example:")
display(best_matches("fee"))
print("\nNon-tie example:")
display(best_matches("reset password"))